In [ ]:
import pandas as pd
import numpy as np
from math import log

# 1. Load data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data"
columns = [
    'class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
    'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
    'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
    'stalk-surface-below-ring', 'stalk-color-above-ring',
    'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number',
    'ring-type', 'spore-print-color', 'population', 'habitat'
]
df = pd.read_csv(url, names=columns)
print(df.head())

# 2. Clean: remove rows with missing values in 'stalk-root'
df = df[df['stalk-root'] != '?']

# 3. Encode categorical variables manually
label_encoders = {}
encoded_df = pd.DataFrame()
for col in df.columns:
    unique_vals = df[col].unique()
    label_encoders[col] = {val: idx for idx, val in enumerate(unique_vals)}
    encoded_df[col] = df[col].map(label_encoders[col])

# 4. Split features and target for the full dataset
X = encoded_df.drop('class', axis=1).values
y = encoded_df['class'].values

# 5. Training Phase: Compute priors and conditional probabilities (with Laplace smoothing)
def train_naive_bayes(X_train, y_train):
    priors = {}
    cond_probs = {}  # cond_probs[c][j][v] holds P(x_j = v | class = c)
    classes = np.unique(y_train)
    n_features = X_train.shape[1]
    for c in classes:
        indices = np.where(y_train == c)[0]
        X_c = X_train[indices]
        priors[c] = len(indices) / len(y_train)
        cond_probs[c] = {}
        for j in range(n_features):
            values, counts = np.unique(X_c[:, j], return_counts=True)
            cond_probs[c][j] = {}
            num_possible = len(np.unique(X_train[:, j]))
            # Laplace smoothing: add 1 to count for each value
            for v, count in zip(values, counts):
                cond_probs[c][j][v] = (count + 1) / (len(X_c) + num_possible)
    return priors, cond_probs

# 6. Prediction Phase: Compute log posterior probabilities and predict the class with highest score
def predict(X_test):
    y_pred = []
    n_features = X_test.shape[1]
    for x in X_test:
        scores = {}
        for c in priors:
            score = log(priors[c])
            for j in range(n_features):
                v = x[j]
                if v in cond_probs[c][j]:
                    score += log(cond_probs[c][j][v])
                else:
                    num_possible = len(np.unique(X_train[:, j]))
                    score += log(1 / (np.sum(y_train == c) + num_possible))
            scores[c] = score
        predicted_class = max(scores, key=scores.get)
        y_pred.append(predicted_class)
    return np.array(y_pred)

# 7. Evaluate on the full dataset with an 80/20 split (here using 60% training as per your code)
split_idx = int(0.6 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
priors, cond_probs = train_naive_bayes(X_train, y_train)
y_pred = predict(X_test)
accuracy = np.mean(y_pred == y_test)
print("Accuracy on full dataset:", accuracy)

# Task 2: Using only 10 categoical attributes 
# Select 10 attributes (including the target 'class')
selected_columns = [
    'class', 'cap-shape', 'cap-surface', 'cap-color', 'odor',
    'gill-size', 'gill-color', 'stalk-root', 'veil-color', 'spore-print-color'
]
df_subset = df[selected_columns]

# Encode the selected features
label_encoders_subset = {}
encoded_df_subset = pd.DataFrame()
for col in df_subset.columns:
    unique_vals = df_subset[col].unique()
    label_encoders_subset[col] = {val: idx for idx, val in enumerate(unique_vals)}
    encoded_df_subset[col] = df_subset[col].map(label_encoders_subset[col])

# Split features and target for the subset
X_subset = encoded_df_subset.drop('class', axis=1).values
y_subset = encoded_df_subset['class'].values

# Train/test split for the subset (using 60% for training)
split_idx_subset = int(0.6 * len(X_subset))
X_train_subset, X_test_subset = X_subset[:split_idx_subset], X_subset[split_idx_subset:]
y_train_subset, y_test_subset = y_subset[:split_idx_subset], y_subset[split_idx_subset:]

# Train Naive Bayes classifier on the subset
priors_subset, cond_probs_subset = train_naive_bayes(X_train_subset, y_train_subset)

# Define a prediction function for the subset (similar to the full-dataset one)
def predict_subset(X_test, X_train, y_train, priors, cond_probs):
    y_pred = []
    n_features = X_test.shape[1]
    for x in X_test:
        scores = {}
        for c in priors:
            score = log(priors[c])
            for j in range(n_features):
                v = x[j]
                if v in cond_probs[c][j]:
                    score += log(cond_probs[c][j][v])
                else:
                    num_possible = len(np.unique(X_train[:, j]))
                    score += log(1 / (np.sum(y_train == c) + num_possible))
            scores[c] = score
        predicted_class = max(scores, key=scores.get)
        y_pred.append(predicted_class)
    return np.array(y_pred)

y_pred_subset = predict_subset(X_test_subset, X_train_subset, y_train_subset, priors_subset, cond_probs_subset)
accuracy_subset = np.mean(y_pred_subset == y_test_subset)
print("Accuracy with 10 features:", accuracy_subset)



  class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill-spacing gill-size gill-color  ... stalk-surface-below-ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk-color-above-ring stalk-color-below-ring veil-type veil-color  \
0                      w                      w         p          w   
1       